<a href="https://colab.research.google.com/github/nashranoor98/credit-card-fraud-detection/blob/main/CaseStudy2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Loading and Studying Dataset

In [ ]:
import pandas as pd

transaction = pd.read_csv("data/train_transaction.csv")
identity = pd.read_csv("data/train_identity.csv")

df = transaction.merge(identity, on="TransactionID", how="left")


In [ ]:
print(df["isFraud"].value_counts())
print(df.shape)


In [ ]:
print(df.head())
print(df.isnull().sum())


2. Splitting Dataset

In [ ]:
from sklearn.model_selection import train_test_split

numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
numeric_columns.remove("isFraud")

fraud_df = df[df["isFraud"] == 1]
normal_df = df[df["isFraud"] == 0].sample(n=min(50000, len(df[df["isFraud"] == 0])), random_state=42)
work_df = pd.concat([normal_df, fraud_df], ignore_index=True)

X = work_df[numeric_columns]
y = work_df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# checking for class distribution
print("Normal Cases: ", sum(y_train == 0))
print("Fraud Cases: ", sum(y_train == 1))


3. Applying SMOTE

In [ ]:
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [ ]:
print("Normal cases: ", sum(y_train_smote == 0))
print("Fraud cases: ", sum(y_train_smote == 1))


4. Training XGBoost and checking Feature Importance


In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt

model = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=2)

model.fit(X_train_smote, y_train_smote)


In [ ]:
feature_importance = pd.Series(model.feature_importances_, index=numeric_columns).sort_values(ascending=False)
print(feature_importance.head(20))

feature_importance.head(20).plot(kind="bar", figsize=(12, 6))
plt.title("Top 20 Feature Importances")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()


5. Predicting and Evaluating Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(roc_auc, 4))


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.10, 0.91, 0.05)
best_threshold = 0.5
best_f1 = 0

for threshold in thresholds:
    y_pred = (y_prob >= threshold).astype(int)
    score = f1_score(y_test, y_pred)
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

y_pred = (y_prob >= best_threshold).astype(int)
print("Selected Threshold:", round(best_threshold, 2))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-Score:", round(f1_score(y_test, y_pred), 4))
